In [12]:
import torch
from pathlib import Path
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-31B-it"
MODEL_DIR = Path("/workspaces/tengoku/models")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    cache_dir=MODEL_DIR,
    attn_implementation="sdpa",
)

# Optimize the model graph for your GPU — first inference will be slow (compile warmup),
# subsequent calls will be significantly faster
model = torch.compile(model, mode="reduce-overhead")

print(f"Loaded {MODEL_ID}")
print(f"Cached to: {MODEL_DIR}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]/workspaces/tengoku/.venv/lib64/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 1188/1188 [00:06<00:00, 180.33it/s]


Loaded google/gemma-4-31B-it
Cached to: /workspaces/tengoku/models
Memory footprint: 17.79 GB


In [11]:
import threading
from transformers import TextIteratorStreamer
from IPython.display import display, Markdown, update_display

def chat(prompt: str, max_new_tokens: int = 1024, thinking: bool = False) -> None:
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking,
    )
    inputs = processor(text=text, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

    thread = threading.Thread(
        target=model.generate,
        kwargs={**inputs, "max_new_tokens": max_new_tokens, "streamer": streamer},
    )
    thread.start()

    display_handle = display(Markdown(""), display_id=True)
    buffer = ""
    for token in streamer:
        buffer += token
        update_display(Markdown(buffer), display_id=display_handle.display_id)

    thread.join()

In [13]:
chat("Hi markdown table me")

Since you didn't specify a topic, I have created a **sample table** for you to show you how they work. 

Here is a table comparing three different types of pets:

| Animal | Diet | Lifespan | Noise Level |
| :--- | :--- | :--- | :--- |
| 🐶 Dog | Omnivore | 10-15 Years | High |
| 🐱 Cat | Carnivore | 12-20 Years | Medium |
| 🐹 Hamster | Omnivore | 2-3 Years | Low |

***

### 🛠️ How to make your own:
If you want to create your own table, use this syntax:

```markdown
| Header 1 | Header 2 | Header 3 |
| :--- | :---: | ---: |
| Left Aligned | Centered | Right Aligned |
| Row 2 Col 1 | Row 2 Col 2 | Row 2 Col 3 |
```

**Quick Tips:**
1. **The Pipe `|`** creates the columns.
2. **The Hyphens `---`** create the header row.
3. **The Colons `:`** control the alignment:
   - `:---` = Left aligned (default)
   - `:---:` = Centered
   - `---:` = Right aligned

**Do you want me to make a specific table for you? Just tell me the topic!**

/workspaces/tengoku/.venv/lib64/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
import gc
from accelerate.hooks import remove_hook_from_module

# Remove accelerate dispatch hooks so .cpu() can move all layers
remove_hook_from_module(model, recurse=True)
model.cpu()

# Clear torch.compile's cached compiled graphs
torch._dynamo.reset()

del model, processor
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")